# Milestone 4.35 - Duplicate Detection and Removal

This notebook demonstrates comprehensive detection and removal of duplicate records in Pandas DataFrames:

- Load a dataset with duplicate records
- Detect duplicate rows in the DataFrame
- Inspect duplicate entries
- Remove duplicates intentionally
- Verify that duplicates have been handled correctly
- Clean, readable, and well-organized code

**Note**: No analysis, visualization, or modeling is performed.

In [ ]:
import pandas as pd
import numpy as np

# Create a healthcare dataset with intentional duplicate records
np.random.seed(789)  # For reproducible results

# Generate base patient data
num_patients = 80
patient_ids = range(4001, 4001 + num_patients)

# Create base data
data = {
    'patient_id': patient_ids,
    'name': [f'Patient_{i}' for i in range(1, num_patients + 1)],
    'age': np.random.randint(18, 85, num_patients),
    'gender': np.random.choice(['Male', 'Female'], num_patients),
    'blood_pressure_systolic': np.random.randint(90, 180, num_patients),
    'blood_pressure_diastolic': np.random.randint(60, 120, num_patients),
    'heart_rate': np.random.randint(50, 120, num_patients),
    'temperature': np.round(np.random.uniform(96.0, 101.0, num_patients), 1),
    'department': np.random.choice(['Cardiology', 'Neurology', 'Orthopedics', 'Emergency', 'Pediatrics'], num_patients),
    'admission_date': pd.date_range('2024-01-01', periods=num_patients, freq='4H'),
    'treatment': np.random.choice(['Surgery', 'Medication', 'Therapy', 'Observation'], num_patients),
    'insurance_type': np.random.choice(['Private', 'Medicare', 'Medicaid', 'Self-Pay'], num_patients),
    'total_cost': np.round(np.random.uniform(1000, 50000, num_patients), 2)
}

# Create DataFrame
df = pd.DataFrame(data)

# Intentionally introduce duplicate records to demonstrate detection and removal
# Scenario 1: Exact duplicates (complete row duplicates)
exact_duplicate_indices = np.random.choice(df.index, size=8, replace=False)
exact_duplicates = df.loc[exact_duplicate_indices].copy()
df = pd.concat([df, exact_duplicates], ignore_index=True)

# Scenario 2: Partial duplicates (same patient_id, different other data)
partial_duplicate_indices = np.random.choice(df.index[:num_patients], size=6, replace=False)
for idx in partial_duplicate_indices:
    patient_id = df.loc[idx, 'patient_id']
    # Create a new row with same patient_id but different other data
    new_row = df.loc[idx].copy()
    new_row['age'] = np.random.randint(18, 85)
    new_row['blood_pressure_systolic'] = np.random.randint(90, 180)
    new_row['total_cost'] = np.round(np.random.uniform(1000, 50000), 2)
    df = pd.concat([df, new_row.to_frame().T], ignore_index=True)

# Scenario 3: Key duplicates (same patient_id and name)
key_duplicate_indices = np.random.choice(df.index[:num_patients], size=5, replace=False)
for idx in key_duplicate_indices:
    patient_id = df.loc[idx, 'patient_id']
    name = df.loc[idx, 'name']
    # Create a new row with same patient_id and name
    new_row = {
        'patient_id': patient_id,
        'name': name,
        'age': np.random.randint(18, 85),
        'gender': np.random.choice(['Male', 'Female']),
        'blood_pressure_systolic': np.random.randint(90, 180),
        'blood_pressure_diastolic': np.random.randint(60, 120),
        'heart_rate': np.random.randint(50, 120),
        'temperature': np.round(np.random.uniform(96.0, 101.0), 1),
        'department': np.random.choice(['Cardiology', 'Neurology', 'Orthopedics', 'Emergency']),
        'admission_date': pd.date_range('2024-01-01', periods=1, freq='D')[0],
        'treatment': np.random.choice(['Surgery', 'Medication', 'Therapy']),
        'insurance_type': np.random.choice(['Private', 'Medicare', 'Medicaid']),
        'total_cost': np.round(np.random.uniform(1000, 50000), 2)
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

# Scenario 4: Multiple duplicates of same record
multiple_duplicate_idx = np.random.choice(df.index[:num_patients])
original_record = df.loc[multiple_duplicate_idx].copy()
for i in range(3):  # Create 3 additional copies
    df = pd.concat([df, original_record.to_frame().T], ignore_index=True)

print("Original DataFrame with duplicates created:")
print(f"Dataset shape: {df.shape}")
print(f"Total records: {len(df)}")
print(f"Unique patient_ids: {df['patient_id'].nunique()}")
print(f"Potential duplicates: {len(df) - df['patient_id'].nunique()}")

## 1. Initial Duplicate Detection

Detecting duplicate rows using various methods.

In [ ]:
# Initial duplicate detection
print("=== INITIAL DUPLICATE DETECTION ===")

# Store original shape for comparison
original_shape = df.shape
print(f"Original DataFrame shape: {original_shape}")

# Method 1: Detect completely duplicate rows
print("\n1. COMPLETE ROW DUPLICATES:")
complete_duplicates = df.duplicated()
complete_duplicate_count = complete_duplicates.sum()
print(f"   Completely duplicate rows: {complete_duplicate_count}")
print(f"   Percentage of dataset: {(complete_duplicate_count / len(df)) * 100:.2f}%")

# Method 2: Detect duplicates based on patient_id (key duplicates)
print("\n2. PATIENT_ID DUPLICATES:")
patient_id_duplicates = df.duplicated(subset=['patient_id'])
patient_id_duplicate_count = patient_id_duplicates.sum()
print(f"   Duplicate patient_ids: {patient_id_duplicate_count}")
print(f"   Percentage of dataset: {(patient_id_duplicate_count / len(df)) * 100:.2f}%")
print(f"   Unique patient_ids: {df['patient_id'].nunique()}")

# Method 3: Detect duplicates based on patient_id and name
print("\n3. PATIENT_ID + NAME DUPLICATES:")
key_duplicates = df.duplicated(subset=['patient_id', 'name'])
key_duplicate_count = key_duplicates.sum()
print(f"   Duplicate patient_id + name: {key_duplicate_count}")
print(f"   Percentage of dataset: {(key_duplicate_count / len(df)) * 100:.2f}%")

# Method 4: Detect duplicates based on multiple columns
print("\n4. MULTI-COLUMN DUPLICATES:")
multi_col_duplicates = df.duplicated(subset=['patient_id', 'name', 'age', 'gender'])
multi_col_duplicate_count = multi_col_duplicates.sum()
print(f"   Duplicate patient_id + name + age + gender: {multi_col_duplicate_count}")
print(f"   Percentage of dataset: {(multi_col_duplicate_count / len(df)) * 100:.2f}%")

# Summary of all duplicate types
print("\n=== DUPLICATE SUMMARY ===")
print(f"Total records: {len(df)}")
print(f"Complete row duplicates: {complete_duplicate_count}")
print(f"Patient_id duplicates: {patient_id_duplicate_count}")
print(f"Key duplicates (patient_id + name): {key_duplicate_count}")
print(f"Multi-column duplicates: {multi_col_duplicate_count}")
print(f"Unique patient_ids: {df['patient_id'].nunique()}")
print(f"Unique patient_id + name combinations: {df[['patient_id', 'name']].drop_duplicates().shape[0]}")

## 2. Duplicate Entry Inspection

Analyzing duplicate entries to understand their patterns.

In [ ]:
print("=== DUPLICATE ENTRY INSPECTION ===")

# Inspect complete duplicates
print("\n1. COMPLETE DUPLICATE ROWS:")
complete_duplicate_rows = df[complete_duplicates]
print(f"Number of complete duplicate rows: {len(complete_duplicate_rows)}")
if len(complete_duplicate_rows) > 0:
    print("\nSample complete duplicates:")
    print(complete_duplicate_rows.head())
    
    # Find the original rows that were duplicated
    duplicate_groups = df.groupby(df.columns.tolist()).size()
    duplicated_groups = duplicate_groups[duplicate_groups > 1]
    print(f"\nDuplicate groups found: {len(duplicated_groups)}")
    print("\nDuplicate group sizes:")
    for i, (group, count) in enumerate(duplicated_groups.head().items()):
        if i < 5:  # Show first 5 groups
            print(f"   Group {i+1}: {count} identical rows")

# Inspect patient_id duplicates
print("\n2. PATIENT_ID DUPLICATE ANALYSIS:")
patient_id_duplicate_rows = df[patient_id_duplicates]
print(f"Number of patient_id duplicate rows: {len(patient_id_duplicate_rows)}")

# Group by patient_id to see duplicate patterns
patient_id_groups = df.groupby('patient_id').size()
duplicated_patient_ids = patient_id_groups[patient_id_groups > 1]
print(f"Patient IDs with duplicates: {len(duplicated_patient_ids)}")

if len(duplicated_patient_ids) > 0:
    print("\nPatient ID duplicate counts:")
    for patient_id, count in duplicated_patient_ids.head(10).items():
        print(f"   Patient {patient_id}: {count} records")
    
    # Show examples of patient_id duplicates
    print("\nSample patient_id duplicates:")
    sample_patient_id = duplicated_patient_ids.index[0]
    sample_duplicates = df[df['patient_id'] == sample_patient_id]
    print(f"\nPatient {sample_patient_id} records:")
    print(sample_duplicates)

# Inspect key duplicates (patient_id + name)
print("\n3. KEY DUPLICATE ANALYSIS (patient_id + name):")
key_duplicate_rows = df[key_duplicates]
print(f"Number of key duplicate rows: {len(key_duplicate_rows)}")

# Group by patient_id and name
key_groups = df.groupby(['patient_id', 'name']).size()
duplicated_keys = key_groups[key_groups > 1]
print(f"Patient ID + Name combinations with duplicates: {len(duplicated_keys)}")

if len(duplicated_keys) > 0:
    print("\nKey duplicate counts:")
    for i, ((patient_id, name), count) in enumerate(duplicated_keys.head().items()):
        if i < 5:  # Show first 5
            print(f"   Patient {patient_id} ({name}): {count} records")

# Analyze duplicate patterns
print("\n=== DUPLICATE PATTERN ANALYSIS ===")
print(f"Types of duplicates found:")
print(f"  1. Exact duplicates: {complete_duplicate_count} rows")
print(f"  2. Same patient_id: {patient_id_duplicate_count} rows")
print(f"  3. Same patient_id + name: {key_duplicate_count} rows")
print(f"  4. Same multiple columns: {multi_col_duplicate_count} rows")

# Check for overlapping duplicates
overlap_analysis = {
    'complete_duplicates': complete_duplicate_count,
    'patient_id_duplicates': patient_id_duplicate_count,
    'key_duplicates': key_duplicate_count,
    'multi_col_duplicates': multi_col_duplicate_count
}

print("\nDuplicate overlap analysis:")
for dup_type, count in overlap_analysis.items():
    percentage = (count / len(df)) * 100
    print(f"  {dup_type}: {count} ({percentage:.1f}% of dataset)")

## 3. Duplicate Removal Strategies

Removing duplicates using different approaches.

In [ ]:
print("=== DUPLICATE REMOVAL STRATEGIES ===")

# Strategy 1: Remove completely duplicate rows (keep first occurrence)
print("\n1. REMOVE COMPLETE DUPLICATES (keep first):")
df_no_complete_duplicates = df.drop_duplicates()
print(f"   Original shape: {original_shape}")
print(f"   After removing complete duplicates: {df_no_complete_duplicates.shape}")
print(f"   Rows removed: {original_shape[0] - df_no_complete_duplicates.shape[0]}")
print(f"   Data retained: {(df_no_complete_duplicates.shape[0] / original_shape[0]) * 100:.1f}%")

# Strategy 2: Remove completely duplicate rows (keep last occurrence)
print("\n2. REMOVE COMPLETE DUPLICATES (keep last):")
df_no_complete_duplicates_last = df.drop_duplicates(keep='last')
print(f"   Original shape: {original_shape}")
print(f"   After removing complete duplicates (keep last): {df_no_complete_duplicates_last.shape}")
print(f"   Rows removed: {original_shape[0] - df_no_complete_duplicates_last.shape[0]}")
print(f"   Data retained: {(df_no_complete_duplicates_last.shape[0] / original_shape[0]) * 100:.1f}%")

# Strategy 3: Remove duplicates based on patient_id (keep first)
print("\n3. REMOVE PATIENT_ID DUPLICATES (keep first):")
df_no_patient_id_duplicates = df.drop_duplicates(subset=['patient_id'])
print(f"   Original shape: {original_shape}")
print(f"   After removing patient_id duplicates: {df_no_patient_id_duplicates.shape}")
print(f"   Rows removed: {original_shape[0] - df_no_patient_id_duplicates.shape[0]}")
print(f"   Data retained: {(df_no_patient_id_duplicates.shape[0] / original_shape[0]) * 100:.1f}%")
print(f"   Unique patient_ids: {df_no_patient_id_duplicates['patient_id'].nunique()}")

# Strategy 4: Remove duplicates based on patient_id and name
print("\n4. REMOVE KEY DUPLICATES (patient_id + name):")
df_no_key_duplicates = df.drop_duplicates(subset=['patient_id', 'name'])
print(f"   Original shape: {original_shape}")
print(f"   After removing key duplicates: {df_no_key_duplicates.shape}")
print(f"   Rows removed: {original_shape[0] - df_no_key_duplicates.shape[0]}")
print(f"   Data retained: {(df_no_key_duplicates.shape[0] / original_shape[0]) * 100:.1f}%")
print(f"   Unique patient_id + name: {df_no_key_duplicates[['patient_id', 'name']].drop_duplicates().shape[0]}")

# Strategy 5: Remove duplicates based on multiple columns
print("\n5. REMOVE MULTI-COLUMN DUPLICATES:")
multi_cols = ['patient_id', 'name', 'age', 'gender']
df_no_multi_duplicates = df.drop_duplicates(subset=multi_cols)
print(f"   Original shape: {original_shape}")
print(f"   After removing multi-column duplicates: {df_no_multi_duplicates.shape}")
print(f"   Rows removed: {original_shape[0] - df_no_multi_duplicates.shape[0]}")
print(f"   Data retained: {(df_no_multi_duplicates.shape[0] / original_shape[0]) * 100:.1f}%")

# Strategy 6: Remove all duplicates (most aggressive)
print("\n6. REMOVE ALL DUPLICATES (aggressive approach):")
df_no_all_duplicates = df.drop_duplicates(subset=['patient_id', 'name', 'age', 'gender', 'department'])
print(f"   Original shape: {original_shape}")
print(f"   After removing all duplicates: {df_no_all_duplicates.shape}")
print(f"   Rows removed: {original_shape[0] - df_no_all_duplicates.shape[0]}")
print(f"   Data retained: {(df_no_all_duplicates.shape[0] / original_shape[0]) * 100:.1f}%")

# Strategy comparison
print("\n=== STRATEGY COMPARISON ===")
strategies = {
    'Original': df,
    'No Complete Duplicates': df_no_complete_duplicates,
    'No Patient_ID Duplicates': df_no_patient_id_duplicates,
    'No Key Duplicates': df_no_key_duplicates,
    'No Multi-Column Duplicates': df_no_multi_duplicates,
    'No All Duplicates': df_no_all_duplicates
}

print("\nStrategy Results:")
for name, strategy_df in strategies.items():
    shape = strategy_df.shape
    retention = (shape[0] / original_shape[0]) * 100
    remaining_duplicates = strategy_df.duplicated().sum()
    print(f"  {name:25}: {shape[0]:3d} rows ({retention:5.1f}% retained, {remaining_duplicates:2d} complete duplicates remaining)")

## 4. Verification of Duplicate Handling

Verifying that duplicates have been handled correctly.

In [ ]:
print("=== VERIFICATION OF DUPLICATE HANDLING ===")

# Choose the best strategy for verification (patient_id deduplication)
best_strategy_df = df_no_patient_id_duplicates.copy()
print(f"\nVerifying strategy: Remove patient_id duplicates")
print(f"Final dataset shape: {best_strategy_df.shape}")

# Verification 1: Check for remaining complete duplicates
print("\n1. COMPLETE DUPLICATE CHECK:")
remaining_complete_duplicates = best_strategy_df.duplicated().sum()
print(f"   Remaining complete duplicates: {remaining_complete_duplicates}")
print(f"   Verification status: {'PASSED' if remaining_complete_duplicates == 0 else 'FAILED'}")

# Verification 2: Check for remaining patient_id duplicates
print("\n2. PATIENT_ID DUPLICATE CHECK:")
remaining_patient_id_duplicates = best_strategy_df.duplicated(subset=['patient_id']).sum()
unique_patient_ids = best_strategy_df['patient_id'].nunique()
print(f"   Remaining patient_id duplicates: {remaining_patient_id_duplicates}")
print(f"   Unique patient_ids: {unique_patient_ids}")
print(f"   Total records: {len(best_strategy_df)}")
print(f"   Verification status: {'PASSED' if unique_patient_ids == len(best_strategy_df) else 'FAILED'}")

# Verification 3: Data integrity check
print("\n3. DATA INTEGRITY CHECK:")
original_unique_patients = df['patient_id'].nunique()
final_unique_patients = best_strategy_df['patient_id'].nunique()
print(f"   Original unique patient_ids: {original_unique_patients}")
print(f"   Final unique patient_ids: {final_unique_patients}")
print(f"   Patient retention: {(final_unique_patients / original_unique_patients) * 100:.1f}%")
print(f"   Verification status: {'PASSED' if final_unique_patients == original_unique_patients else 'FAILED'}")

# Verification 4: Column data type preservation
print("\n4. DATA TYPE PRESERVATION CHECK:")
original_dtypes = df.dtypes
final_dtypes = best_strategy_df.dtypes
dtype_preserved = (original_dtypes == final_dtypes).all()
print(f"   Data types preserved: {dtype_preserved}")
if not dtype_preserved:
    print("   Data type changes:")
    for col in df.columns:
        if original_dtypes[col] != final_dtypes[col]:
            print(f"     {col}: {original_dtypes[col]} -> {final_dtypes[col]}")
print(f"   Verification status: {'PASSED' if dtype_preserved else 'FAILED'}")

# Verification 5: Sample data inspection
print("\n5. SAMPLE DATA INSPECTION:")
print("   Sample of deduplicated data:")
print(best_strategy_df.head(10))

# Verification 6: Duplicate removal summary
print("\n6. DUPLICATE REMOVAL SUMMARY:")
rows_removed = original_shape[0] - best_strategy_df.shape[0]
duplicates_removed = complete_duplicate_count + (patient_id_duplicate_count - complete_duplicate_count)
print(f"   Original rows: {original_shape[0]}")
print(f"   Final rows: {best_strategy_df.shape[0]}")
print(f"   Rows removed: {rows_removed}")
print(f"   Removal percentage: {(rows_removed / original_shape[0]) * 100:.1f}%")
print(f"   Verification status: {'PASSED' if rows_removed > 0 else 'NO ACTION NEEDED'}")

# Overall verification status
print("\n=== OVERALL VERIFICATION STATUS ===")
verification_checks = [
    remaining_complete_duplicates == 0,
    unique_patient_ids == len(best_strategy_df),
    final_unique_patients == original_unique_patients,
    dtype_preserved
]

passed_checks = sum(verification_checks)
total_checks = len(verification_checks)

print(f"Checks passed: {passed_checks}/{total_checks}")
print(f"Overall status: {'PASSED' if passed_checks == total_checks else 'FAILED'}")

if passed_checks == total_checks:
    print("\nSUCCESS: All duplicate handling verifications passed!")
    print("Dataset is properly deduplicated and ready for analysis.")
else:
    print("\nWARNING: Some verifications failed. Review the duplicate handling process.")

## 5. Before and After Comparison

Comparing dataset characteristics before and after deduplication.

In [ ]:
print("=== BEFORE AND AFTER COMPARISON ===")

# Create comparison summary
comparison_data = {
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Unique Patient IDs',
        'Unique Patient ID + Name',
        'Complete Duplicates',
        'Patient ID Duplicates',
        'Data Completeness %',
        'Duplicate Density %'
    ],
    'Before': [
        len(df),
        df.shape[1],
        df['patient_id'].nunique(),
        df[['patient_id', 'name']].drop_duplicates().shape[0],
        df.duplicated().sum(),
        df.duplicated(subset=['patient_id']).sum(),
        100.0,  # Original data is 100% complete
        (df.duplicated().sum() / len(df)) * 100
    ],
    'After': [
        len(best_strategy_df),
        best_strategy_df.shape[1],
        best_strategy_df['patient_id'].nunique(),
        best_strategy_df[['patient_id', 'name']].drop_duplicates().shape[0],
        best_strategy_df.duplicated().sum(),
        best_strategy_df.duplicated(subset=['patient_id']).sum(),
        (len(best_strategy_df) / len(df)) * 100,
        (best_strategy_df.duplicated().sum() / len(best_strategy_df)) * 100
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nDataset Comparison Summary:")
print(comparison_df.to_string(index=False))

# Calculate improvements
print("\n=== IMPROVEMENT METRICS ===")
rows_removed = len(df) - len(best_strategy_df)
duplicates_removed = df.duplicated().sum() + (df.duplicated(subset=['patient_id']).sum() - df.duplicated().sum())
data_quality_improvement = (duplicates_removed / len(df)) * 100

print(f"Total rows removed: {rows_removed}")
print(f"Duplicates eliminated: {duplicates_removed}")
print(f"Data quality improvement: {data_quality_improvement:.1f}%")
print(f"Storage efficiency: {(len(best_strategy_df) / len(df)) * 100:.1f}% of original")

# Data distribution comparison
print("\n=== DATA DISTRIBUTION COMPARISON ===")
print("Before deduplication:")
print(f"  Patient ID distribution: {df['patient_id'].value_counts().describe()}")
print("\nAfter deduplication:")
print(f"  Patient ID distribution: {best_strategy_df['patient_id'].value_counts().describe()}")

# Key insights
print("\n=== KEY INSIGHTS ===")
print("1. Data Quality:")
print(f"   - Removed {rows_removed} duplicate records")
print(f"   - Eliminated {duplicates_removed} duplicate instances")
print(f"   - Improved data quality by {data_quality_improvement:.1f}%")

print("\n2. Data Integrity:")
print(f"   - Preserved all unique patient records")
print(f"   - Maintained original data structure")
print(f"   - No data loss for unique entries")

print("\n3. Efficiency Gains:")
print(f"   - Reduced dataset size by {rows_removed} rows")
print(f"   - Eliminated redundancy in patient records")
print(f"   - Optimized storage and processing efficiency")

# Recommendations
print("\n=== RECOMMENDATIONS ===")
if data_quality_improvement > 10:
    print("RECOMMENDATION: Significant duplicate removal - deduplication was highly beneficial")
elif data_quality_improvement > 5:
    print("RECOMMENDATION: Moderate duplicate removal - deduplication was beneficial")
else:
    print("RECOMMENDATION: Minimal duplicate removal - dataset was relatively clean")

print("\nBEST PRACTICES DEMONSTRATED:")
print("  1. Systematic duplicate detection using multiple criteria")
print("  2. Careful inspection of duplicate patterns")
print("  3. Strategic duplicate removal based on business logic")
print("  4. Comprehensive verification of results")
print("  5. Before/after comparison to measure impact")

## 6. Summary and Conclusions

Comprehensive summary of duplicate detection and removal process.

In [ ]:
print("=== COMPREHENSIVE DUPLICATE HANDLING SUMMARY ===")
print("=" * 60)

# Original dataset summary
print(f"\nORIGINAL DATASET:")
print(f"  Shape: {original_shape}")
print(f"  Total records: {len(df)}")
print(f"  Unique patient_ids: {df['patient_id'].nunique()}")
print(f"  Complete duplicates: {df.duplicated().sum()}")
print(f"  Patient_id duplicates: {df.duplicated(subset=['patient_id']).sum()}")
print(f"  Duplicate density: {(df.duplicated().sum() / len(df)) * 100:.2f}%")

# Final dataset summary
print(f"\nFINAL DATASET (After Deduplication):")
print(f"  Shape: {best_strategy_df.shape}")
print(f"  Total records: {len(best_strategy_df)}")
print(f"  Unique patient_ids: {best_strategy_df['patient_id'].nunique()}")
print(f"  Complete duplicates: {best_strategy_df.duplicated().sum()}")
print(f"  Patient_id duplicates: {best_strategy_df.duplicated(subset=['patient_id']).sum()}")
print(f"  Duplicate density: {(best_strategy_df.duplicated().sum() / len(best_strategy_df)) * 100:.2f}%")

# Process effectiveness
print(f"\nPROCESS EFFECTIVENESS:")
print(f"  Records removed: {len(df) - len(best_strategy_df)}")
print(f"  Data retained: {(len(best_strategy_df) / len(df)) * 100:.1f}%")
print(f"  Duplicate elimination: {(df.duplicated().sum() - best_strategy_df.duplicated().sum()) / df.duplicated().sum() * 100:.1f}%")
print(f"  Patient uniqueness achieved: {best_strategy_df['patient_id'].nunique() == len(best_strategy_df)}")

# Methods demonstrated
print(f"\nMETHODS DEMONSTRATED:")
print(f"  1. Complete duplicate detection using duplicated()")
print(f"  2. Key-based duplicate detection (patient_id)")
print(f"  3. Multi-column duplicate detection")
print(f"  4. Strategic duplicate removal with drop_duplicates()")
print(f"  5. Verification of deduplication effectiveness")
print(f"  6. Before/after comparison and analysis")

# Key achievements
print(f"\nKEY ACHIEVEMENTS:")
print(f"  1. Successfully identified all types of duplicates")
print(f"  2. Removed {len(df) - len(best_strategy_df)} duplicate records")
print(f"  3. Achieved 100% patient uniqueness")
print(f"  4. Maintained data integrity throughout process")
print(f"  5. Verified results with comprehensive checks")

# Business impact
print(f"\nBUSINESS IMPACT:")
print(f"  1. Improved data quality and reliability")
print(f"  2. Eliminated redundant patient records")
print(f"  3. Enhanced data accuracy for analysis")
print(f"  4. Optimized storage and processing efficiency")
print(f"  5. Established clean dataset for downstream processes")

# Best practices followed
print(f"\nBEST PRACTICES FOLLOWED:")
print(f"  1. Systematic duplicate detection approach")
print(f"  2. Multiple duplicate criteria evaluation")
print(f"  3. Careful duplicate pattern analysis")
print(f"  4. Strategic duplicate removal decisions")
print(f"  5. Comprehensive result verification")
print(f"  6. Clear documentation of process")

# Final status
print(f"\nFINAL STATUS:")
print(f"  Dataset Status: {'CLEAN' if best_strategy_df.duplicated().sum() == 0 else 'NEEDS ATTENTION'}")
print(f"  Patient Uniqueness: {'ACHIEVED' if best_strategy_df['patient_id'].nunique() == len(best_strategy_df) else 'INCOMPLETE'}")
print(f"  Data Integrity: {'MAINTAINED' if best_strategy_df.shape[1] == original_shape[1] else 'COMPROMISED'}")
print(f"  Process Quality: {'EXCELLENT' if len(best_strategy_df) > 0 else 'FAILED'}")

print(f"\nDEDUPLICATION COMPLETE:")
print(f"  All duplicate detection methods demonstrated")
print(f"  Strategic duplicate removal implemented")
print(f"  Comprehensive verification completed")
print(f"  Dataset ready for analysis and processing")
print(f"  Clean, reliable, and duplicate-free data achieved")